Data Acquisition for Ground-Truth for Image Prediction

In [ ]:
!pip install geemap python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.4 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import datetime
import time
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()  

GEE_PROJECT     = os.getenv("GEE_PROJECT",     "thesis-478211")
GEE_ASSET_PATH  = os.getenv("GEE_ASSET_PATH",  "projects/thesis-478211/assets/BC_Barangays")
DRIVE_FOLDER    = os.getenv("DRIVE_FOLDER",    "DW_Q1_Colorized")
START_YEAR      = int(os.getenv("START_YEAR",  "2016"))
END_YEAR        = int(os.getenv("END_YEAR",    str(datetime.datetime.now().year)))
SCALE           = int(os.getenv("SCALE",       "10"))
MAX_PIXELS      = float(os.getenv("MAX_PIXELS", "1e13"))

print(f"Project      : {GEE_PROJECT}")
print(f"Asset path   : {GEE_ASSET_PATH}")
print(f"Drive folder : {DRIVE_FOLDER}")
print(f"Year range   : {START_YEAR} – {END_YEAR}")
print(f"Scale        : {SCALE} m  |  Max pixels: {MAX_PIXELS:.0e}")

In [ ]:


ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

In [ ]:
boundary = ee.FeatureCollection(GEE_ASSET_PATH)

dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(boundary)

VIS_PALETTE = [
    '419bdf','397d49','88b053','7a87c6','e49635',
    'dfc35a','c4281b','a59b8f','b39fe1'
]

def q1_dw_rgb(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 3, 31)

    filtered = dw.filterDate(start, end).select("label")

    count = filtered.size().getInfo()
    if count == 0:
        return None

    img_mode = filtered.mode().clip(boundary)
    img_rgb = img_mode.visualize(min=0, max=8, palette=VIS_PALETTE)
    img_rgb = img_rgb.set("year", year).set("quarter", 1)

    return img_rgb

exports = []

for year in range(START_YEAR, END_YEAR + 1):
    print(f"Processing {year} Q1 ...")

    img_rgb = q1_dw_rgb(year)

    if img_rgb is None:
        print(f"Skipping {year} Q1 — no DW data")
        continue

    description = f"DW_RGB_{year}_Q1"

    task = ee.batch.Export.image.toDrive(
        image=img_rgb,
        description=description,
        folder=DRIVE_FOLDER,
        fileNamePrefix=description,
        region=boundary.geometry(),
        scale=SCALE,
        maxPixels=MAX_PIXELS
    )

    task.start()
    exports.append((description, task))
    print(f"Started export: {description}")

    while True:
        status = task.status()
        state = status['state']

        if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
            break
        print(f"  Task status: {state} ... waiting 30s")
        time.sleep(30)

    if state == 'COMPLETED':
        print(f"Export successful: {description}")
    else:
        print(f"Export {state}: {description}")

print("\n=== ALL Q1 EXPORTS FINISHED ===")
for desc, task in exports:
    status = task.status()['state']
    print(f"{desc}: {status}")

Processing 2016 Q1 ...
Started export: DW_RGB_2016_Q1
  Task status: READY ... waiting 30s
  Task status: RUNNING ... waiting 30s
Export successful: DW_RGB_2016_Q1
Processing 2017 Q1 ...
Started export: DW_RGB_2017_Q1
  Task status: READY ... waiting 30s
  Task status: RUNNING ... waiting 30s
Export successful: DW_RGB_2017_Q1
Processing 2018 Q1 ...
Started export: DW_RGB_2018_Q1
  Task status: READY ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNNING ... waiting 30s
Export successful: DW_RGB_2018_Q1
Processing 2019 Q1 ...
Started export: DW_RGB_2019_Q1
  Task status: READY ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNNING ... waiting 30s
Export successful: DW_RGB_2019_Q1
Processing 2020 Q1 ...
Started export: DW_RGB_2020_Q1
  Task status: READY ... waiting 30s
  Task status: RUNNING ... waiting 30s
  Task status: RUNN